In [2]:
###### 250910 미션 ######

# 종합 과제: “충성 사용자 & 영화 인사이트 리포트”영화 서비스의 이용자·댓글·영화 데이터를 바탕으로, 
# 마케팅팀이 바로 사용할 수 있는 두 가지 리포트를 만들어 주세요. 결과는 각각 별도의 출력 구조로 제출합니다.
# 1) 충성 사용자 리포트
# • 이용자와 댓글 데이터를 결합해, 총 댓글 수가 10개 이상인 이용자만 추립니다.
# • 각 이용자에 대해 다음 지표를 계산해 주세요:
#     ◦ commentsCount(총 댓글 수)
#     ◦ avgTextLen(댓글 평균 길이, null/missing은 길이 0으로 간주)
#     ◦ lastCommentDate(가장 최근 댓글 시점)
# • 결과에는 **이름(name), 이메일(email)**만 식별 정보로 포함하고, 
# 위 활동 지표 **내림차순(많이·길게·최근 순)**으로 정렬해 상위 이용자가 누구인지 한눈에 보이도록 합니다.
# 2) 영화 인사이트 리포트
# • 영화 컬렉션만으로 한 번의 파이프라인에서 다음을 동시에 산출해 주세요.
#     1. 최신 영화 5편: 제목(title)과 연도(year) 목록
#     2. 고평점 영화 개수: imdb.rating ≥ 8 작품의 총 개수(숫자 1개)
#     3. 장르별 상위 10: 장르별 작품 수를 집계해 상위 10개를 많이 나온 순으로 정렬
#     4. 연도별 평균 평점(최근 10년): 최신 연도를 기준으로 최근 10개 연도의 평균 평점을 산출(연도-오름차순 표기)출력 형식 가이드
# • 충성 사용자 리포트: 리스트(배열) 형태. 각 원소는 {name, email, commentsCount, avgTextLen, lastCommentDate}.
# • 영화 인사이트 리포트: 단일 문서(오브젝트) 형태.
#     ◦ 키는 latest5, highRatedCount, genresTop10, yearlyAvgRecent10로 구성.
#     ◦ highRatedCount는 숫자 하나로 제출5:13위 문제는 Jupyter Notebook & pymongo 활용하여 문제 해결 및 ".ipynb" 확장자로 제출!!!

In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")

db = client.sample_mflix
movies = db.movies


pipeline = [
    {
        "$lookup": {
            "from": "comments",
            "localField": "email",
            "foreignField": "email",
            "as": "userComments"
        }
    },
    {
        "$addFields": {
            "commentsCount": { "$size": "$userComments" },
            "avgTextLen": {
                "$avg": {
                    "$map": {
                        "input": "$userComments",
                        "as": "c",
                        "in": { "$strLenCP": { "$ifNull": ["$$c.text", ""] } }
                    }
                }
            },
            "lastCommentDate": { "$max": "$userComments.date" } 
        }
    },
    {
        "$match": { "commentsCount": { "$gte": 10 } }
    },
    {
        "$project": {
            "_id": 0,
            "name": 1,
            "email": 1,
            "commentsCount": 1,
            "avgTextLen": 1,
            "lastCommentDate": 1
        }
    },
    {
        "$sort": {
            "commentsCount": -1,
            "avgTextLen": -1,
            "lastCommentDate": -1
        }
    }
]

result = db.users.aggregate(pipeline)
for data in result:
    print(data)

{'name': 'Mace Tyrell', 'email': 'roger_ashton-griffiths@gameofthron.es', 'commentsCount': 331, 'avgTextLen': 152.49244712990938, 'lastCommentDate': datetime.datetime(2017, 8, 2, 1, 37, 26)}
{'name': 'Missandei', 'email': 'nathalie_emmanuel@gameofthron.es', 'commentsCount': 327, 'avgTextLen': 153.0948012232416, 'lastCommentDate': datetime.datetime(2017, 9, 11, 16, 52, 51)}
{'name': 'The High Sparrow', 'email': 'jonathan_pryce@gameofthron.es', 'commentsCount': 315, 'avgTextLen': 152.68253968253967, 'lastCommentDate': datetime.datetime(2017, 7, 10, 4, 58, 23)}
{'name': 'Sansa Stark', 'email': 'sophie_turner@gameofthron.es', 'commentsCount': 308, 'avgTextLen': 153.42532467532467, 'lastCommentDate': datetime.datetime(2017, 5, 2, 16, 29, 50)}
{'name': 'Rodrik Cassel', 'email': 'ron_donachie@gameofthron.es', 'commentsCount': 305, 'avgTextLen': 151.75737704918032, 'lastCommentDate': datetime.datetime(2017, 9, 2, 6, 1, 19)}
{'name': 'Thoros of Myr', 'email': 'paul_kaye@gameofthron.es', 'commen